# 📡 01 - Multi-Modal Ingestion: RSS Feeds & YouTube Audio

### Pipeline Stage 1: Data Ingestion & Normalization
This notebook demonstrates our dual-modality ingestion engine:
1. **Live and pre-cached RSS feeds** spanning 10 institutional, retail, and tech media outlets.
2. **YouTube audio extraction and local Whisper transcription** for financial influencer video content.

---

In [ ]:
import pandas as pd
from src.ingest import RSSIngester, YouTubeIngester, ingest_all

# Ingest sample multimodal dataset (offline reproducible mode)
df_raw = ingest_all(use_sample=True)
print(f'Total items ingested: {len(df_raw)}')
print(f'Modality breakdown:\n{df_raw["source_type"].value_counts()}')
df_raw.head(3)

## 2. Outlet Representation & Volume Analysis

Let's inspect how articles are distributed across media categories and bias labels.

In [ ]:
outlet_summary = df_raw.groupby(['outlet', 'category', 'bias_label']).size().reset_index(name='article_count')
outlet_summary.sort_values(by='article_count', ascending=False)

## 3. Multi-Modal Audio Ingestion: Local Whisper Transcription

Financial hype frequently spreads via video and podcast channels before hitting print. Our audio pipeline ingests YouTube audio tracks via `yt-dlp` and generates transcriptions locally using OpenAI Whisper without sending any data to third-party cloud APIs.

In [ ]:
yt_sample = YouTubeIngester.load_cached_transcript()
print(f'Video Title:        {yt_sample["title"]}')
print(f'Channel:            {yt_sample["channel"]}')
print(f'Speaker:            {yt_sample["speaker"]}')
print(f'Local Whisper Model:{yt_sample["whisper_model_used"]}')
print(f'Duration:           {yt_sample["duration_sec"]} seconds')
print('\n--- Transcript Excerpt ---')
print(yt_sample['full_transcript'][:400] + '...')
print('\n--- First 3 Timestamped Audio Segments ---')
for seg in yt_sample['segments'][:3]:
    print(f'[{seg["start"]:04.1f}s - {seg["end"]:04.1f}s] {seg["text"]}')

## 4. Persist Ingested Articles to Interim Cache

Save raw ingested articles to `data/interim/ingested_articles.csv` for downstream feature extraction.

In [ ]:
from src.config import INTERIM_DATA_DIR

interim_file = INTERIM_DATA_DIR / 'ingested_articles.csv'
df_raw.to_csv(interim_file, index=False)
print(f'✓ Successfully saved {len(df_raw)} records to {interim_file}')
print('Proceed to Notebook 02 for feature engineering and hype scoring!')